# Pipeline smoke test

Run the full pipeline once, inspect the DQ report, and query silver via DuckDB. Safe to re-run — writes are idempotent.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

# In Colab: !pip install -q -r ../requirements.txt
# In Colab: os.environ['EIA_API_KEY'] = 'your-key'
# In Colab: os.environ['DATA_ROOT'] = '/content/drive/MyDrive/collide/data'


In [ ]:
from orchestrator.run_once import main as run_once
run_once([])  # all sources

In [ ]:
from pipeline.config import load_config
import duckdb, pandas as pd
cfg = load_config()
con = duckdb.connect(str(cfg.data_root / '_meta' / 'catalog.duckdb'), read_only=True)
con.execute('SHOW TABLES').fetchall()

In [ ]:
# AZPS demand vs forecast for the last 48h — sanity plot
df = con.execute("""
  SELECT period_utc, type, value_mw
  FROM eia930
  WHERE respondent='AZPS' AND type IN ('D','DF')
  ORDER BY period_utc
""").df()
df.pivot(index='period_utc', columns='type', values='value_mw').plot(figsize=(12,4), title='AZPS demand vs day-ahead forecast')

In [ ]:
# Check DQ report
import json, glob
latest = sorted(glob.glob(str(cfg.data_root / '_meta' / 'runs' / '*.json')))[-1]
print(json.dumps(json.loads(open(latest).read()), indent=2)[:3000])